In [ ]:
#test- data1124 - ceshi - ceshi1125 - 1126tiaoshi

In [ ]:
######主要在搭建模型（前向-后向）
######test- data1124 - ceshi - ceshi1125 - 1126tiaoshi

######chuban(qianxiangchuanbo)

import torch
import torch.nn as nn

class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(8, 16), 
            nn.Tanh(),
            nn.Linear(16, 1)   
        )
        

        self.decoder = nn.Sequential(
            nn.Linear(1, 16),
            nn.Tanh(),
            nn.Linear(16, 8)  )
        

        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth  
        self.quant_min = -2.0
        self.quant_max = 2.0

    def quantize(self, x):

        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        x_normalized = (x - self.quant_min) / scale

        return torch.round(torch.clamp(x_normalized, 0, self.levels - 1)).long().squeeze()

    def quantized_to_binary(self, quantized):

        binary = torch.zeros(self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):

            binary[self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary

    def binary_to_quantized(self, binary):

        quantized = torch.tensor(0, dtype=torch.long, device=binary.device)
        for bit in binary:
            quantized = quantized * 2 + bit 
        return quantized

    def dequantize(self, x_quantized):

        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        input_shape = x.shape
        
        x_flat = x.view(-1)  
        x_encoded = self.encoder(x_flat)  
        

        x_quantized = self.quantize(x_encoded) 
        

        x_binary = self.quantized_to_binary(x_quantized)  
        
        x_quantized_recovered = self.binary_to_quantized(x_binary)
        
        x_dequantized = self.dequantize(x_quantized_recovered).unsqueeze(0)  
        
        x_decoded_flat = self.decoder(x_dequantized)
        x_output = x_decoded_flat.view(input_shape)  
        
        return x_output, x_quantized, x_binary


if __name__ == "__main__":
    input_data = torch.randn(2, 4, dtype=torch.float32)
    print("输入数据形状:", input_data.shape)
    print("输入数据:\n", input_data)
    
    codec = CodecSystem()
    

    output_data, quantized_data, binary_data = codec(input_data)

    print("\n量化等级（0-15，对应4比特）:", quantized_data.item())
    print("4比特二进制表示:", binary_data.numpy()) 
    print("二进制形状:", binary_data.shape) 
    print("\n输出数据形状:", output_data.shape)
    print("输出数据:\n", output_data)
    
    # 验证一致性
    assert input_data.shape == output_data.shape
    
    recovered_quantized = codec.binary_to_quantized(binary_data)
    assert recovered_quantized == quantized_data



In [ ]:
######1119 基础版(qianxiang+houxiang)
import torch
import torch.nn as nn
import torch.optim as optim

seed = 42 
torch.manual_seed(seed)

# 正确实现的自定义量化函数，使用 torch.autograd.Function
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        
        x_normalized = (x - quant_min) / scale
        
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()
        
        
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(8, 16),  
            nn.Tanh(),
            nn.Linear(16, 1)  
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(1, 16),
            nn.Tanh(),
            nn.Linear(16, 8)  
        )
        
     
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -1.0
        self.quant_max = 1.0
        
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)

    def quantized_to_binary(self, quantized):
        if quantized.dim() == 0: 
            quantized = quantized.unsqueeze(0)
            
        binary = torch.zeros(quantized.numel(), self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary.squeeze(0) 

    def binary_to_quantized(self, binary):
        if binary.dim() == 1: 
            binary = binary.unsqueeze(0)
            
        quantized = torch.zeros(binary.shape[0], dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized.squeeze(0)

    def dequantize(self, x_quantized):
        if x_quantized.dim() == 0:
            x_quantized = x_quantized.unsqueeze(0)
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        input_shape = x.shape
        

        x_flat = x.view(-1, 8) 
        x_encoded = self.encoder(x_flat) 
        

        x_quantized = self.quantizer(x_encoded) 
        

        x_binary = self.quantized_to_binary(x_quantized)
        
        x_quantized_recovered = self.binary_to_quantized(x_binary)
        
        x_dequantized = self.dequantize(x_quantized_recovered) 
        x_dequantized = x_dequantized.unsqueeze(1) 

        x_decoded_flat = self.decoder(x_dequantized)
        x_output = x_decoded_flat.view(input_shape)  
        
        return x_output, x_quantized, x_binary


if __name__ == "__main__":

    codec = CodecSystem()
    criterion = nn.MSELoss()  
    optimizer = optim.Adam(codec.parameters(), lr=0.001)
    

    batch_size = 100
    train_data = torch.randn(batch_size, 2, 4, dtype=torch.float32)
    
   
    num_epochs = 1000
    codec.train()  
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        
    
        outputs, _, _ = codec(train_data)
        
      
        loss = criterion(outputs, train_data)
        
       
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.6f}')

    print("\n--- 训练后验证 ---")
    codec.eval() 
    test_input = torch.randn(1, 2, 4) 
    print("原始输入:\n", test_input.squeeze(0))
    
    with torch.no_grad(): 
        output_data, quantized_data, binary_data = codec(test_input)
    
    print("\n量化等级 (0-15):", quantized_data.item())
    print("4比特二进制表示:", binary_data.numpy())
    print("解码输出:\n", output_data.squeeze(0))
    
    final_loss = criterion(output_data, test_input)
    print(f"\n最终重构误差 (MSE): {final_loss.item():.6f}")



In [ ]:

######zhongban  +youhuaqi?
import torch
import torch.nn as nn
import torch.optim as optim

seed = 42
torch.manual_seed(seed)

# 
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        # 创建mask：只在有效量化范围内传递梯度,防止超出量化范围的数值影响训练   QAT?
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float() #应用mask到梯度
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.Tanh(),  
            nn.Linear(16, 8),
            nn.Tanh(),
            nn.Linear(8, 1),
            #nn.Tanh()
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(1, 8),
            nn.Tanh(),
            nn.Linear(8, 16),
            nn.Tanh(),
            nn.Linear(16, 8),
            #nn.Tanh()
        )
        
        # 4比特量化配置
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2
        self.quant_max = 2
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)

        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))

    def quantized_to_binary(self, quantized):
        if quantized.dim() == 0:
            quantized = quantized.unsqueeze(0)
        binary = torch.zeros(quantized.numel(), self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary.squeeze(0)

    def binary_to_quantized(self, binary):
        if binary.dim() == 1:
            binary = binary.unsqueeze(0)
        quantized = torch.zeros(binary.shape[0], dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized.squeeze(0)

    def dequantize(self, x_quantized):
        if x_quantized.dim() == 0:
            x_quantized = x_quantized.unsqueeze(0)
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        input_shape = x.shape
        x_flat = x.view(-1, 8)
        
       
        x_encoded = self.encoder(x_flat)
       
        x_encoded = x_encoded * self.output_scale + self.output_bias
        
        # 量化
        x_quantized = self.quantizer(x_encoded)
        x_binary = self.quantized_to_binary(x_quantized)
        x_quantized_recovered = self.binary_to_quantized(x_binary)
        
        # 解量化
        x_dequantized = self.dequantize(x_quantized_recovered)
        x_dequantized = x_dequantized.unsqueeze(1)
        
        # 解码
        x_decoded_flat = self.decoder(x_dequantized)
        x_output = x_decoded_flat.view(input_shape)
        
        return x_output, x_quantized, x_binary

# 
if __name__ == "__main__":
  
    codec = CodecSystem()
    criterion = nn.MSELoss()
    
    optimizer = optim.Adam(codec.parameters(), lr=0.0005)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000, eta_min=0.0001)
    
  
    num_train_samples = 500
    batch_size = 300
    train_data = torch.randn(num_train_samples, 2, 4, dtype=torch.float32)
    

    num_epochs = 1000
    codec.train()
    
    print("开始量化专用优化训练...")
    for epoch in range(num_epochs):
        indices = torch.randperm(num_train_samples)[:batch_size]
        batch_data = train_data[indices]
        
        optimizer.zero_grad()
        outputs, _, _ = codec(batch_data)
        loss = criterion(outputs, batch_data)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=0.5)
        
        optimizer.step()
        scheduler.step()
        
        if (epoch + 1) % 100 == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.6f}, LR: {current_lr:.6f}')

    
    codec.eval()
    
  
    num_test_samples = 200
    test_data = torch.randn(num_test_samples, 2, 4, dtype=torch.float32)
    
    total_loss = 0.0
    
    #使用torch.no_grad()来加速验证过程，避免计算梯度
    with torch.no_grad():
       
        for i in range(num_test_samples):
            test_input = test_data[i:i+1]  
            output_data, _, _ = codec(test_input)
            
           
            loss = criterion(output_data, test_input)
            total_loss += loss.item()

 
    average_test_loss = total_loss / num_test_samples
    print(f"所有测试样本的平均MSE: {average_test_loss:.6f}")